In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# name
str_function_name = 'genxii-lgd-update-feats'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    
    # load output from concat sensitivity
    print('Loading output from sensitivity analysis concatenation...')
    str_filename = 'df_sensitivity.csv'
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/02_model/05_lambda_concat_sensitivity/{str_filename}'
    df = pd.read_csv(str_uri)
    
    # get the top feature
    print('Getting feature that helps the model most once removed...')
    str_col = df['feature'].iloc[0]
    
    # import the features to drop
    print('Importing features to drop...')
    str_filename = 'df_feats_to_drop.csv'
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    list_cols_drop = list(pd.read_csv(str_uri)['feature'])
    
    # append
    print(f'Appending {str_col} to {str_uri}...')
    list_cols_drop.append(str_col)
    
    # create df and upload to s3
    print(f'Creating data frame and uploading to {str_uri}...')
    df = pd.DataFrame({'feature': list_cols_drop})
    df.to_csv(str_uri, index=False)

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-lgd-update-feats

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  14.34kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> 54a294f880ad
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 96a84c45543c
Step 3/6 : COPY requirements.txt  .
 ---> Using cache
 ---> f3798621783a
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> d4fa9067d383
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> 9fca311093e1
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in 89745f580131
Removing intermediate container 89745f580131
 ---> 18caab53f152
Successfully built 18caab53f152
Successfully tagged genxii-lgd-update-feats:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
{
    "repository": {
        "repositoryArn": "arn:aws:ecr:us-west-2:836690756591:repository/genxii-lgd-update-feats",
        "registryId": "836690756591",
        "repositoryName": "genxii-lgd-update-feats",
        "repositoryUri": "836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-lgd-update-feats",
        "createdAt": 1697728397.0,
        "imageTagMutability": "MUTABLE",
        "imageScanningConfiguration": {
            "scanOnPush": true
        },
        "encryptionConfiguration": {
            "encryptionType": "AES256"
        }
    }
}
The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-lgd-update-feats]
a47748e8b671: Preparing
39c85950c1a6: Preparing
3fe351c8d0ea: Preparing
895fcc9c35e0: Preparing
2ae7c19e0b5c: Preparing
9e6784b558c3: Preparing
15dd6c63f3a2: Preparing
8819b61d0672: Preparing
30349c0bf45d: Preparing
91232f425615: Preparing
8819b61d0672: Waiting
91232f425615: Waiting
9e6784b558c3: Waiting
15dd6c63f3a2: Wai

### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

In [10]:
# create function
str_image_uri = '836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-lgd-update-feats:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': 'f8b0f5f5e154290529112012b47af19e131d6c9c2c357af1d64430a29b5b6ee5',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-update-feats',
 'FunctionName': 'genxii-lgd-update-feats',
 'LastModified': '2023-10-19T15:13:36.073+0000',
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1066',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 19 Oct 2023 15:13:36 GMT',
                                      'x-amzn-requestid': 'ed3695c0-d419-4097-8ac6-e5d2c26081fc'},
                      'HTTPStatusCode': 201,
                      'RequestId': 'ed3695c0-d419-4097-8ac6-e5d2c26081fc',
                      'RetryAttempts': 0},
 'RevisionId': '1169bda3-9ec6-414b-8075-b4

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)